# ElasticNet con la probabilidad de CatBoost

Experimento rápido para comprobar si añadir la probabilidad continua de CatBoost mejora la estimación de `bdi_total` en puntos BDI-II. Se comparan dos ElasticNet con los mismos hiperparámetros y las mismas seis variables. La segunda versión añade una séptima variable.

Se reutilizan la partición y los hiperparámetros finales del proyecto. Cada probabilidad del entrenamiento se obtiene fuera de pliegue, con un CatBoost que no entrenó con ese participante. El test se usa al final para comparar las dos versiones. Los hiperparámetros se fijaron previamente con el entrenamiento del proyecto, por lo que este experimento rápido no repite la búsqueda en cada pliegue.

Ejecuta con el entorno de Jupyter del proyecto. Requiere el CSV original, `split_indices.csv` y los dos modelos finales guardados.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, r2_score, recall_score, root_mean_squared_error
from sklearn.model_selection import cross_val_predict

actual = Path.cwd()
raiz = next((p for p in (actual, *actual.parents) if (p / "src" / "Modelado.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abre el notebook desde el repositorio.")
sys.path[:0] = [str(raiz / "src"), str(raiz)]

from data.load_data import CORTE_BDI, cargar_datos
from Modelado import PREDICTORES, RUTA_SALIDA, SEED, TARGET, cargar_modelo_final, cargar_split, configurar_dispositivo, crear_cv
from ModeladoRegresion import TARGET_REG, cargar_modelo_regresion

if not (RUTA_SALIDA / "split_indices.csv").is_file():
    raise FileNotFoundError("Falta split_indices.csv. Genera la partición del proyecto desde Modelado.ipynb.")

df = cargar_datos()
train, test = cargar_split(df)
if (len(train) + len(test) != len(df) or
    not df["subject_id"].is_unique or
    not set(train["subject_id"]).isdisjoint(test["subject_id"])):
    raise ValueError("La partición debe cubrir el CSV y separar a los participantes.")

X_train, X_test = train[PREDICTORES], test[PREDICTORES]
y_train, y_test = train[TARGET_REG], test[TARGET_REG]
y_train_clf = train[TARGET]

clf_final, meta_clf = cargar_modelo_final()
reg_final, meta_reg = cargar_modelo_regresion()
if meta_clf["modelo"] != "catboost" or meta_reg["modelo"] != "elasticnet" or meta_reg["variante_target"] != "nativo":
    raise ValueError("El experimento espera los modelos finales CatBoost y ElasticNet nativo del proyecto.")

print(f"Entrenamiento: {len(train)} participantes · Test: {len(test)} participantes")

Entrenamiento: 3848 participantes · Test: 962 participantes


## 1. Probabilidades fuera de pliegue

Cinco CatBoost generan las probabilidades del entrenamiento. Para las probabilidades del test se ajusta otro CatBoost con todo el entrenamiento. En ambos casos se usa la probabilidad continua, con el preprocesamiento dentro del pipeline.

In [2]:
clf = configurar_dispositivo(clone(clf_final), gpu=False)
clf.set_params(modelo__thread_count=4, modelo__allow_writing_files=False)
prob_train = cross_val_predict(
    clf, X_train, y_train_clf, cv=crear_cv(SEED), method="predict_proba", n_jobs=1,
)[:, 1]

clf.fit(X_train, y_train_clf)
prob_test = clf.predict_proba(X_test)[:, 1]
X_train_ampliado = X_train.assign(prob_catboost=prob_train)
X_test_ampliado = X_test.assign(prob_catboost=prob_test)

## 2. Comparación con los mismos hiperparámetros

Se reajustan los dos regresores con el objetivo nativo, `bdi_total`. El preprocesador de la versión ampliada escala también la probabilidad. Entre los casos cercanos al corte se incluyen los puntajes observados de 11 a 17.

In [3]:
base = clone(reg_final)
ampliado = clone(reg_final)
preprocesador = ampliado.named_steps["preprocesamiento"]
transformadores = [
    (nombre, transf, [*columnas, "prob_catboost"] if nombre == "numericas" else columnas)
    for nombre, transf, columnas in preprocesador.transformers
]
preprocesador.set_params(transformers=transformadores)

base.fit(X_train, y_train)
ampliado.fit(X_train_ampliado, y_train)
predicciones = {
    "ElasticNet": base.predict(X_test),
    "ElasticNet + prob. CatBoost": ampliado.predict(X_test_ampliado),
}

cerca = y_test.between(CORTE_BDI - 3, CORTE_BDI + 3).to_numpy()
positivos = y_test.to_numpy() >= CORTE_BDI
alerta_catboost = prob_test >= meta_clf["umbral"]

filas = {}
for nombre, pred in predicciones.items():
    corte_estimado = pred >= CORTE_BDI
    filas[nombre] = {
        "RMSE": root_mean_squared_error(y_test, pred),
        "MAE": mean_absolute_error(y_test, pred),
        "R²": r2_score(y_test, pred),
        "MAE cerca de 14": mean_absolute_error(y_test[cerca], pred[cerca]) if cerca.any() else np.nan,
        "Sesgo cerca de 14": (pred[cerca] - y_test[cerca]).mean() if cerca.any() else np.nan,
        "Recall del corte estimado (%)": 100 * recall_score(positivos, corte_estimado, zero_division=0),
        "Desacuerdo con alerta CatBoost (%)": 100 * (corte_estimado != alerta_catboost).mean(),
    }
tabla = pd.DataFrame.from_dict(filas, orient="index")
tabla.loc["Cambio · ampliado menos base"] = tabla.iloc[1] - tabla.iloc[0]
print(f"Casos del test cerca de 14: {cerca.sum()} · Umbral de alerta CatBoost: {meta_clf['umbral']:.2%}")
display(tabla.round(4))

Casos del test cerca de 14: 122 · Umbral de alerta CatBoost: 15.34%


,RMSE,MAE,R²,MAE cerca de 14,Sesgo cerca de 14,Recall del corte estimado (%),Desacuerdo con alerta CatBoost (%)
ElasticNet,6.5951,4.7262,0.2323,4.3178,-3.6584,17.8344,34.8233
ElasticNet + prob. CatBoost,6.5830,4.7115,0.2351,4.4151,-3.6381,19.7452,34.1996
Cambio · ampliado menos base,-0.0121,-0.0146,0.0028,0.0974,0.0203,1.9108,-0.6237


## Cómo leer la comparación

Un cambio negativo en RMSE o MAE indica menor error. Un sesgo negativo cerca de 14 indica que el modelo tiende a subestimar el puntaje en ese intervalo. El recall mide cuántos casos observados con BDI-II ≥ 14 llegan también a 14 en el puntaje estimado. La última columna compara dos decisiones distintas y se reporta como diagnóstico del desacuerdo. La decisión de añadir la variable se basa en el error del puntaje y en el comportamiento cerca de 14.

Este notebook compara ambos enfoques en la partición existente del proyecto. Para elegir nuevos hiperparámetros o confirmar una mejora, usa validación dentro del entrenamiento y reserva datos nuevos para una evaluación final independiente.